# Finance task previews

Random samples for each finance task from the MTBench finance data.
Change `N` or `SEED` to see a different subset.

| Task | Data | Evaluation script |
|---|---|---|
| Time series forecasting | aligned pairs | `value_prediction.py --indicator time` |
| Trend prediction | aligned pairs | `trend_classification.py` |
| Technical indicators (MACD, Bollinger) | aligned pairs | `value_prediction.py --indicator macd/bb` |
| Correlation | `QAshort` / `QAlong` | `correlation_prediction.py` |
| Multiple-choice QA | `QAshort` / `QAlong` | `mcqa.py` |

In [ ]:
import json
import random
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.append("..")
from evaluation.finance.meta_prompt import (
    finance_correlation_metaprompt_generation,
    finance_mcqa_metaprompt_generation,
)

DATA = Path("../data/processed/finance")
N = 3
SEED = 0

# setting -> aligned-pairs folder and QA folder
ALIGNED = {
    "short (7 days in, 1 day out)": "aligned_in7days_out1days",
    "long (30 days in, 7 days out)": "aligned_in30days_out7days",
}
QA = {"short": "QAshort", "long": "QAlong"}


def load_aligned(folder):
    df = pd.read_parquet(DATA / folder / "data" / "train-00000-of-00001.parquet")
    for col in ["text", "trend", "technical", "alignment"]:
        df[col] = df[col].map(json.loads)
    return df


def sample_rows(df, n=N, seed=SEED):
    return df.loc[random.Random(seed).sample(list(df.index), n)]


def sample_files(folder, n=N, seed=SEED):
    files = sorted((DATA / folder).glob("*.json"))
    return random.Random(seed).sample(files, n)


def load(path):
    with open(path) as f:
        return json.load(f)


def short(text, n=400):
    text = " ".join(str(text).split())
    return text if len(text) <= n else text[:n] + " ..."


def when(ts):
    return datetime.fromtimestamp(ts).strftime("%Y-%m-%d %H:%M")


def plot_row(df, in_key, out_key, title_fn, in_label="input", out_label="output", n=N):
    fig, axes = plt.subplots(1, len(df), figsize=(5 * len(df), 3), squeeze=False)
    for ax, (idx, row) in zip(axes[0], df.iterrows()):
        x_in, x_out = in_key(row), out_key(row)
        ax.plot(range(len(x_in)), x_in, label=in_label)
        ax.plot(range(len(x_in), len(x_in) + len(x_out)), x_out, label=out_label)
        ax.set_title(title_fn(idx, row), fontsize=8)
        ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()


aligned = {setting: load_aligned(folder) for setting, folder in ALIGNED.items()}

## Aligned-pair tasks

Time series forecasting, trend prediction and technical indicators all read the same aligned files
(`input_window`, `output_window`, `text`, `trend`, `technical`), so one plot per sample is shown with each task's label.

### Time series forecasting

In [ ]:
for setting, df in aligned.items():
    print(f"{'=' * 100}\nTime series forecasting | {setting} | {len(df):,} samples")
    rows = sample_rows(df)
    plot_row(
        rows,
        lambda r: r["input_window"],
        lambda r: r["output_window"],
        lambda i, r: f"row {i}: {when(r['input_timestamps'][0])[:10]} -> {when(r['output_timestamps'][-1])[:10]}",
    )
    for idx, row in rows.iterrows():
        print(f"[row {idx}] predict {len(row['output_window'])} prices from {len(row['input_window'])} inputs")
        print("   news:", short(row["text"]["content"]))

### Trend prediction

Label is the 5-way bin of the output window's percentage change (`<-4%`, `-2% ~ -4%`, `-2% ~ +2%`, `+2% ~ +4%`, `>+4%`).

In [ ]:
for setting, df in aligned.items():
    print(f"{'=' * 100}\nTrend prediction | {setting}")
    rows = sample_rows(df)
    plot_row(
        rows,
        lambda r: r["input_window"],
        lambda r: r["output_window"],
        lambda i, r: f"row {i}: {r['trend']['output_bin_label']} ({r['trend']['output_percentage_change']:+.1f}%)",
    )
    for idx, row in rows.iterrows():
        t = row["trend"]
        print(
            f"[row {idx}] output: {t['output_bin_label']} ({t['output_percentage_change']:+.2f}%), "
            f"input: {t['input_bin_label']}, alignment: {row['alignment']}"
        )
        print("   news:", short(row["text"]["content"], 300))

### Technical indicators

MACD and the upper Bollinger band are predicted over the output window, from the input window's values.

In [ ]:
for setting, df in aligned.items():
    for name, in_col, out_col in [("MACD", "in_macd", "out_macd"), ("Upper Bollinger band", "in_upper_bb", "out_upper_bb")]:
        print(f"{'=' * 100}\nTechnical indicator: {name} | {setting}")
        rows = sample_rows(df)
        plot_row(
            rows,
            lambda r: r["technical"][in_col],
            lambda r: r["technical"][out_col],
            lambda i, r: f"row {i}: {name}",
            in_label=f"input {name}",
            out_label=f"target {name}",
        )
        for idx, row in rows.iterrows():
            out = row["technical"][out_col]
            print(f"[row {idx}] {len(out)} target values, first {out[0]:.3f}, last {out[-1]:.3f}")

## Correlation

Predict how strongly the price movement correlates with the news sentiment (5 classes).

In [ ]:
for setting in QA:
    print(f"{'=' * 100}\nCorrelation | {setting} ({QA[setting]})")
    files = sample_files(QA[setting])
    fig, axes = plt.subplots(1, len(files), figsize=(5 * len(files), 3), squeeze=False)
    for ax, path in zip(axes[0], files):
        d = load(path)
        n_in = len(d["input_window"])
        ax.plot(range(n_in), d["input_window"], label="input")
        ax.plot(range(n_in, n_in + len(d["output_window"])), d["output_window"], label="output")
        ax.set_title(f"{path.stem}: {d['news_price_correlation']}", fontsize=8)
        ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()
    for path in files:
        d = load(path)
        print(f"[{path.name}] correlation: {d['news_price_correlation']} (news published {d['published_utc']})")
        print("   news:", short(d["text"], 300))

## Multiple-choice QA

In [ ]:
for setting in QA:
    print(f"{'=' * 100}\nMCQA | {setting} ({QA[setting]})")
    for path in sample_files(QA[setting]):
        d = load(path)
        print(f"[{path.name}]")
        print("   news:", short(d["text"], 300))
        print("   question:", d["MCQA"]["question"].replace("\n", "\n             "))
        print("   answer:", d["MCQA"]["answer"])

### Rendered prompts (series truncated)

One sample per setting showing the prompts `correlation_prediction.py` and `mcqa.py` build.

In [ ]:
for setting in QA:
    path = sample_files(QA[setting], 1)[0]
    d = load(path)
    args = dict(
        setting=setting,
        sticker=path.name.split("_")[1].split(".")[0],
        time1=datetime.fromtimestamp(d["input_timestamps"][0]),
        time2=datetime.fromtimestamp(d["input_timestamps"][-1]),
        in_price=d["input_window"][:4] + ["..."],
        news=short(d["text"], 200),
        time_news=d["published_utc"],
    )
    print(f"--- correlation | {setting} ({path.name}) ---\n{finance_correlation_metaprompt_generation(**args)}\n")
    print(f"--- MCQA | {setting} ({path.name}) ---\n{finance_mcqa_metaprompt_generation(**args, question=d['MCQA']['question'])}\n")

## Dataset sizes

In [ ]:
for setting, df in aligned.items():
    print(f"aligned pairs, {setting:32s} {len(df):>6,} samples")
for setting, folder in QA.items():
    print(f"QA {setting:8s} ({folder:8s}) {len(list((DATA / folder).glob('*.json'))):>6,} samples")